# 03_extract_instance_features.ipynb

根据 train/val/test splits 产生 instance-level 特征和标签

执行顺序：从上到下逐个 cell 运行

In [1]:
# Cell 1: 导入必要的库
import yaml
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import sys

from src.data.feature_extractor import process_one_clip

d:\Python_env\Dolphin\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Cell 2: 加载配置文件
def load_config(config_path: str = "configs/default.yaml") -> dict:
    path = Path(config_path)
    if not path.exists():
        print(f"配置文件不存在：{path}", file=sys.stderr)
        sys.exit(1)
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)

cfg = load_config()
print("配置文件加载成功")

配置文件加载成功


In [3]:
# Cell 3: 读取路径和参数
paths = {k: Path(v) for k, v in cfg["paths"].items()}
preprocess = cfg["preprocess"]
extract_cfg = cfg.get("extract", {})

# 打印关键路径，方便检查
print("clips_dir:", paths["clips_dir"])
print("clip_label_csv:", paths["clip_label_csv"])
print("splits_dir:", paths["splits_dir"])
print("features_root:", paths["features_root"])

clips_dir: data\processed\clips_1min
clip_label_csv: data\processed\metadata\clip_labels.csv
splits_dir: data\processed\splits
features_root: data\processed\features_1s


In [4]:
# Cell 4: 创建输出目录
for d in [
    paths["features_root"] / paths["npy_subdir"],
    paths["features_root"] / paths["img_subdir"],
    paths["instance_label_csv"].parent
]:
    d.mkdir(parents=True, exist_ok=True)

print("输出目录已创建/确认存在")

输出目录已创建/确认存在


In [5]:
# Cell 5: 读取数据
clip_df = pd.read_csv(paths["clip_label_csv"])
anno_df = pd.read_csv(paths["anno_csv"])
anno_df["Ori_file_num(No.)"] = anno_df["Ori_file_num(No.)"].astype(int)

print(f"clip_df 行数：{len(clip_df)}")
print(f"anno_df 行数：{len(anno_df)}")

clip_df 行数：145
anno_df 行数：832


In [6]:
# Cell 6: 获取需要处理的 clip stems
def get_needed_stems(splits_dir: Path) -> set:
    needed = set()
    for name in ["train", "val", "test"]:
        p = splits_dir / f"{name}.txt"
        if p.exists():
            with p.open("r", encoding="utf-8") as f:
                needed.update(line.strip() for line in f if line.strip())
        else:
            print(f"警告：split 文件不存在，已跳过：{p}")
    return needed

needed_stems = get_needed_stems(paths["splits_dir"])
print(f"需要处理的 clip stem 数量：{len(needed_stems)}")

需要处理的 clip stem 数量：87


In [7]:
# Cell 7: 过滤 clip_df
process_df = clip_df[
    clip_df["clip_filename"].str.replace(".wav", "", regex=False).isin(needed_stems)
]

if process_df.empty:
    print("过滤后没有 clip 需要处理，退出")
else:
    print(f"将处理 {len(process_df)} / {len(clip_df)} 个 clip")

将处理 87 / 145 个 clip


In [8]:
# Cell 8: 合并 config（包含所有路径和预处理参数）
full_config = {
    **preprocess,
    "clips_dir": str(paths["clips_dir"]),
    "output_root": str(paths["features_root"]),
    "npy_dir": str(paths["npy_subdir"]),
    "img_dir": str(paths["img_subdir"]),
    "dpi": preprocess.get("dpi", 150),
}

print("full_config 预览：", {k: v for k, v in list(full_config.items())[:8]})

full_config 预览： {'target_sr': 48000, 'sr': 48000, 'clip_duration_sec': 60.0, 'filename_pad_zero': 3, 'audio_subtype': 'FLOAT', 'discard_short_tail': True, 'label_in_filename': True, 'clips_root': 'data/processed/clips_1min'}


In [10]:
# Cell 9: 核心提取循環（簡化版，已移除 instance_overlaps 相關內容）

all_records = []

DEBUG_CLIP_LIMIT = 0   # 設為 0 = 不印細節；設為 3 = 只印前 3 個 clip 的基本資訊

clip_count = 0

for row in tqdm(
    process_df.itertuples(index=False),
    total=len(process_df),
    desc="提取 instance-level 特征"
):
    clip_count += 1
    clip_name = row.clip_filename
    orig_audio = row.original_audio
    clip_start_sec = row.start_sec

    # 只在 DEBUG 模式下印一些基本資訊
    if DEBUG_CLIP_LIMIT > 0 and clip_count <= DEBUG_CLIP_LIMIT:
        print(f"\n{'='*80}")
        print(f"處理第 {clip_count} 個 clip: {clip_name}")
        print(f"  來自原始音頻: {orig_audio}")
        print(f"  這個 clip 在原音頻中的時間區間: [{clip_start_sec:.3f}s, {clip_start_sec + 60:.3f}s)")

    try:
        # 只接收 records（單一返回值）
        records = process_one_clip(row._asdict(), anno_df, full_config)

        # 可選：簡單統計這個 clip 產生了多少正樣本
        if DEBUG_CLIP_LIMIT > 0 and clip_count <= DEBUG_CLIP_LIMIT:
            pos_in_clip = sum(1 for r in records if r["label"] == 1)
            total_in_clip = len(records)
            print(f"  → 產生 {total_in_clip} 個 instance，其中正樣本 {pos_in_clip} 個")

        all_records.extend(records)

        if DEBUG_CLIP_LIMIT > 0 and clip_count == DEBUG_CLIP_LIMIT:
            print(f"\n已顯示前 {DEBUG_CLIP_LIMIT} 個 clip 的基本資訊，剩餘 clip 只顯示進度條")

    except Exception as e:
        print(f"處理 clip {clip_name} 時出錯：{e}")
        if extract_cfg.get("error_on_missing_clip", True):
            raise

print("\n提取完成")
print(f"總共產生 {len(all_records)} 條 instance 記錄")

提取 instance-level 特征: 100%|██████████| 87/87 [11:41<00:00,  8.07s/it]


提取完成
總共產生 5220 條 instance 記錄


In [11]:
# Cell 10: 保存结果并统计
instance_df = pd.DataFrame(all_records)
instance_df.to_csv(paths["instance_label_csv"], index=False)

print(f"\n完成！共产生 {len(instance_df)} 条记录")
print(f"保存至：{paths['instance_label_csv']}")

if not instance_df.empty:
    pos_ratio = instance_df["label"].mean()
    pos_count = instance_df["label"].sum()
    total = len(instance_df)
    print(f"实例级正样本比例：{pos_ratio:.4f} ({pos_count} / {total})")


完成！共产生 5220 条记录
保存至：data\processed\instance_labels\instance_labels.csv
实例级正样本比例：0.0511 (267 / 5220)
